In [2]:
import sys
import os
package_path = os.path.abspath("..")
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from dask.distributed import Client, LocalCluster

In [5]:
cluster=LocalCluster()
client=Client(cluster)

2025-06-20 14:20:34,580 - distributed.diskutils - INFO - Found stale lock file and directory '/var/folders/1d/0w58jwl11g91yrp_p3jm9fdw0000gn/T/dask-worker-space/worker-tpovf0aq', purging
2025-06-20 14:20:34,580 - distributed.diskutils - INFO - Found stale lock file and directory '/var/folders/1d/0w58jwl11g91yrp_p3jm9fdw0000gn/T/dask-worker-space/worker-rtv2ehkd', purging
2025-06-20 14:20:34,580 - distributed.diskutils - INFO - Found stale lock file and directory '/var/folders/1d/0w58jwl11g91yrp_p3jm9fdw0000gn/T/dask-worker-space/worker-5l9dydc5', purging
2025-06-20 14:20:34,581 - distributed.diskutils - INFO - Found stale lock file and directory '/var/folders/1d/0w58jwl11g91yrp_p3jm9fdw0000gn/T/dask-worker-space/worker-l_hu0dm3', purging
2025-06-20 14:20:34,581 - distributed.diskutils - INFO - Found stale lock file and directory '/var/folders/1d/0w58jwl11g91yrp_p3jm9fdw0000gn/T/dask-worker-space/worker-i7oi92x4', purging
2025-06-20 14:20:34,581 - distributed.diskutils - INFO - Found st

In [ ]:
#load some data, filter, and 
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/shendure/shendure_counts_grouped.txt")
filtered,dropped_groups=scm.filter_low_umi_count(dat)
dropped_groups

In [20]:
#run first time & cache on disc. If it's not the first time, load from disc instead. 
first_run=False

primordial=None

if first_run:
    primordial=scm.ortho()
    primordial.criss_cross(client=client,dat=filtered)
    primordial.extract_params(client)
    primordial.save('/gpfs/gibbs/pi/reilly/tabula_data/shendure','ortho')
else:
    primordial=scm.ortho.load(client,'/gpfs/gibbs/pi/reilly/tabula_data/shendure','ortho')

In [ ]:
#create a simulation batch object
batch=scm.simulation_batch(primordial)
batch.describe_primordial(client,filtered)

In [23]:
batch.simulate_many(client,3)

TypeError: Trying to convert dd.Scalar<monoton..., dtype=bool> to a boolean value. Because Dask objects are lazily evaluated, they cannot be converted to a boolean value or used in boolean conditions like if statements. Try calling .compute() to force computation prior to converting to a boolean value or using in a conditional statement.

In [ ]:
batch.fit_to_simulations(client)

In [ ]:
batch._flatten_all_parameters()

In [ ]:
batch.plot_nb_spread()

In [1]:
cluster.close()

NameError: name 'cluster' is not defined